# Topic 3: Stateless Transformations

> **Phase 6 → Week 10 → Topic 3**

---

## Stateless vs Stateful

```
Stateless:                              Stateful:
  Each row is processed independently.    Requires memory of past rows.
  No cross-row or cross-batch state.      Maintains state across micro-batches.

  Examples:                               Examples:
    filter(col > 100)                       groupBy().count()
    withColumn(new_col, expr)               window aggregations
    select(col1, col2)                      stream-stream joins
    map / flatMap                           deduplication
    explode()
    join with a STATIC DataFrame

  Output mode: APPEND always works        Output mode: UPDATE or COMPLETE
  State store: NOT required               State store: REQUIRED
  Memory: bounded (safe)                  Memory: grows without watermark!
```

---

## Stream-Static Join

```
A stream joined with a STATIC DataFrame is stateless:
  - The static side is loaded once (or re-loaded each batch)
  - Each incoming row from the stream is enriched using the static lookup
  - No cross-batch state needed

  streaming_events.join(static_users, "user_id")  ← STATELESS — works with append

Stream-stream join is STATEFUL (covered in Week 11).
```

---

## Interview Cheat Sheet

**Q: Can you use UDFs in Structured Streaming?**
> Yes. Stateless UDFs work exactly like in batch — applied per row. Since each row is independent, there is no state to manage. The same performance advice applies: prefer built-in functions, use pandas UDFs for vectorized operations. The UDF must be deterministic (same input = same output), otherwise the exactly-once guarantee breaks.

**Q: Can you join a stream with a static DataFrame?**
> Yes — this is a common enrichment pattern. The stream provides the incoming events; the static DataFrame provides a lookup table (e.g., product catalog, user table). Spark loads the static side once and broadcasts it to all executors. The join is stateless because each incoming row can be matched without knowledge of past rows. Supports append output mode.

**Q: What transformations are NOT supported in Structured Streaming?**
> `sort()` on unbounded streams (can only sort within windows), `limit()`, `distinct()` without watermark, multiple streaming aggregations chained together. Also, operations that require all data to be present (like a global sort) cannot work on an infinite stream.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("Week10 - Stateless Transformations") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:49:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready


## Part 1: filter, select, withColumn — Row-by-Row Stateless

In [2]:
# Simulate a stream of order events using rate source
# value mod 100 → order amount (1–99)
# value mod 5  → customer bucket

order_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 20) \
    .load() \
    .withColumn("order_id",    F.concat(F.lit("O"), F.lpad(F.col("value").cast("string"), 6, "0"))) \
    .withColumn("amount",      (F.col("value") % 100 + 1).cast("double")) \
    .withColumn("customer_id", F.concat(F.lit("C"), (F.col("value") % 5 + 1).cast("string"))) \
    .withColumn("category",    F.element_at(
        F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
        (F.col("value") % 3 + 1).cast("int"))) \
    .drop("value")

# Stateless transformations — same API as batch
high_value = order_stream \
    .filter(F.col("amount") > 50) \
    .withColumn("tax",   F.round(F.col("amount") * 0.18, 2)) \
    .withColumn("total", F.round(F.col("amount") + F.col("tax"), 2)) \
    .select("order_id", "customer_id", "category", "amount", "tax", "total")

q = high_value.writeStream \
    .format("memory") \
    .queryName("high_value_orders") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(8)
print("High-value orders (amount > 50):")
spark.sql("""
    SELECT category, count(*) as orders, round(avg(amount),2) as avg_amount
    FROM high_value_orders
    GROUP BY category
    ORDER BY category
""").show()

q.stop()

26/05/16 08:49:43 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e8f12430-758d-4690-93d4-a5986c4c5c1a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:49:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/05/16 08:49:46 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 2000 milliseconds, but spent 2152 milliseconds


High-value orders (amount > 50):


+-----------+------+----------+
|   category|orders|avg_amount|
+-----------+------+----------+
|   clothing|    16|      75.5|
|electronics|    17|      76.0|
|       food|    17|      75.0|
+-----------+------+----------+



## Part 2: explode() — One Row to Many

In [3]:
# Simulate a stream where each row is a JSON order with multiple items
# We model this by creating an array column and exploding it

basket_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load() \
    .withColumn("order_id",  F.concat(F.lit("O"), F.col("value").cast("string"))) \
    .withColumn("items", F.array(
        F.struct(F.lit("item_A").alias("sku"), F.lit(10.0).alias("price")),
        F.struct(F.lit("item_B").alias("sku"), F.lit(25.0).alias("price")),
        F.struct(F.lit("item_C").alias("sku"), (F.col("value") % 50 + 5).cast("double").alias("price")),
    )) \
    .drop("value")

# Explode: one row per order → one row per item
line_items = basket_stream \
    .withColumn("item", F.explode(F.col("items"))) \
    .select(
        F.col("order_id"),
        F.col("item.sku").alias("sku"),
        F.col("item.price").alias("price"),
    )

q2 = line_items.writeStream \
    .format("memory") \
    .queryName("line_items") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(8)
print("Line items after explode (3 per order):")
spark.sql("""
    SELECT sku, count(*) as line_count, round(avg(price), 2) as avg_price
    FROM line_items
    GROUP BY sku ORDER BY sku
""").show()

q2.stop()

26/05/16 08:49:55 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8593b3d6-e849-442e-994d-f22f0335598f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:49:55 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Line items after explode (3 per order):


+------+----------+---------+
|   sku|line_count|avg_price|
+------+----------+---------+
|item_A|        30|     10.0|
|item_B|        30|     25.0|
|item_C|        30|     19.5|
+------+----------+---------+



26/05/16 08:50:04 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@7cba162] is aborting.
26/05/16 08:50:04 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@7cba162] aborted.


## Part 3: Stream-Static Join — Enrichment Pattern

In [4]:
# Static lookup table (e.g., product catalog, user table from a database)
product_catalog = spark.createDataFrame([
    ("electronics", "Electronics",   0.18, "EL"),
    ("clothing",    "Fashion",        0.12, "CL"),
    ("food",        "Grocery & Food", 0.05, "FD"),
], ["category", "department", "tax_rate", "dept_code"])

# Stream with category column
event_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("order_id",  F.concat(F.lit("O"), F.col("value").cast("string"))) \
    .withColumn("amount",    (F.col("value") % 500 + 10).cast("double")) \
    .withColumn("category", F.element_at(
        F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
        (F.col("value") % 3 + 1).cast("int"))) \
    .drop("value")

# Stream-static join: enriches each streaming row with catalog data
enriched = event_stream.join(
    product_catalog,            # static side — loaded once, broadcasted
    on="category",
    how="left"
).withColumn("tax_amount",  F.round(F.col("amount") * F.col("tax_rate"), 2)) \
 .withColumn("grand_total", F.round(F.col("amount") + F.col("tax_amount"), 2)) \
 .select("order_id", "category", "department", "dept_code",
         "amount", "tax_amount", "grand_total")

q3 = enriched.writeStream \
    .format("memory") \
    .queryName("enriched_orders") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(8)
print("Enriched orders (stream joined with static catalog):")
spark.sql("""
    SELECT department, dept_code, count(*) as orders,
           round(sum(amount),0) as revenue, round(sum(tax_amount),0) as tax
    FROM enriched_orders
    GROUP BY department, dept_code
    ORDER BY department
""").show()

q3.stop()

26/05/16 08:50:04 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:267)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:43)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.$anonfun$run$5(WriteToDataSourceV2Exec.scala:446)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1397)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.run(WriteToDataSourceV2Exec.s

26/05/16 08:50:05 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-58544a6f-7bd2-40ad-8509-477301c4e434. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:50:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Enriched orders (stream joined with static catalog):


+--------------+---------+------+-------+-----+
|    department|dept_code|orders|revenue|  tax|
+--------------+---------+------+-------+-----+
|   Electronics|       EL|    20|  770.0|139.0|
|       Fashion|       CL|    20|  790.0| 95.0|
|Grocery & Food|       FD|    20|  810.0| 41.0|
+--------------+---------+------+-------+-----+



## Part 4: UDFs in Streaming

In [5]:
from pyspark.sql.functions import udf

# UDFs work identically in streaming — same registration, same call
# Rule: UDF must be deterministic (same input → same output)

@udf(StringType())
def classify_amount(amount):
    """Categorize order amount into tiers."""
    if amount is None:
        return "unknown"
    if amount < 50:
        return "small"
    elif amount < 200:
        return "medium"
    else:
        return "large"

@udf(StringType())
def mask_customer_id(cid):
    """Mask customer ID for PII compliance."""
    if cid is None:
        return None
    return cid[:2] + "***"

udf_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("amount",      (F.col("value") % 300 + 1).cast("double")) \
    .withColumn("customer_id", F.concat(F.lit("CUST"), (F.col("value") % 10).cast("string"))) \
    .withColumn("tier",        classify_amount(F.col("amount"))) \
    .withColumn("masked_cid",  mask_customer_id(F.col("customer_id"))) \
    .select("timestamp", "amount", "tier", "masked_cid")

q4 = udf_stream.writeStream \
    .format("memory") \
    .queryName("udf_results") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(6)
print("UDF results:")
spark.sql("""
    SELECT tier, masked_cid, count(*) as cnt, round(avg(amount),1) as avg_amount
    FROM udf_results GROUP BY tier, masked_cid
    ORDER BY tier
""").show(10)

q4.stop()

26/05/16 08:50:14 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6a1026fb-1197-441c-9518-36dfaba3a65e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:50:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


UDF results:


+-----+----------+---+----------+
| tier|masked_cid|cnt|avg_amount|
+-----+----------+---+----------+
|small|     CU***| 30|      15.5|
+-----+----------+---+----------+



## Exercises

1. Create a streaming pipeline that: reads from `rate`, generates a `payload` column as a JSON string (`{"id": value, "score": value % 10}`), then parses it back using `from_json()`. This simulates receiving JSON events from Kafka.
2. Add a streaming-static join where you join a stream of `user_id` events with a static DataFrame of `user_id → user_name` mappings. What happens to rows where the `user_id` is not in the static table?
3. Write a stateless UDF that extracts the domain from an email string (e.g., `"alice@gmail.com"` → `"gmail.com"`). Apply it to a streaming column. What happens if the email is null or malformed?
4. Why can't you call `df.sort()` on an unbounded stream? What is the workaround?
5. You have a streaming join with a static product table. The product table is updated daily in the database. How do you ensure the streaming job picks up the latest version of the static table?